# RAG Evaluation

This notebook evaluates the end-to-end RAG pipeline by retrieving relevant products and generating recommendations using the retrieved product information and customer reviews.

## 1. Setup

The retrieval and generation components used by the application are loaded to evaluate the complete RAG pipeline.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval import HybridRetriever
from src.context_builder import build_context
from src.gemini_generator import GeminiGenerator

pd.set_option("display.max_colwidth", 100)

retriever = HybridRetriever()
generator = GeminiGenerator()

Loading product data...
Loading BM25 index...
Loading FAISS index...
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hybrid retriever ready.


## 2. Evaluation Queries

Four representative queries are used to evaluate the end-to-end RAG pipeline across different product requirements and constraints.

In [3]:
queries = [
    "wireless earbuds with good battery life",
    "comfortable earbuds for running with good battery life",
    "waterproof wireless earbuds for sports",
    "iPhone case with good grip"
]

queries

['wireless earbuds with good battery life',
 'comfortable earbuds for running with good battery life',
 'waterproof wireless earbuds for sports',
 'iPhone case with good grip']

## 3. Retrieved Products

The hybrid retriever is used to retrieve the top five products for each evaluation query. These products form the evidence passed to the RAG generation stage.

In [4]:
retrieved_results = {}

for query in queries:
    results = retriever.search(query, top_k=5)
    retrieved_results[query] = results

    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)

    display(
        pd.DataFrame([
            {
                "Rank": rank,
                "Product ID": result["parent_asin"],
                "Title": result["title"],
                "Rating": result["average_rating"],
                "Reviews": result["review_count"]
            }
            for rank, result in enumerate(results, start=1)
        ])
    )

QUERY: wireless earbuds with good battery life


,Rank,Product ID,Title,Rating,Reviews
0,1,B08H15SQ3H,Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stere...,4.0,76
1,2,B0C778Z3RJ,"Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time-C...",3.9,80
2,3,B07NP3KSN2,"TREBLAB X5 - High-End Bluetooth Earbuds w/Beryllium Speakers - True HD Sound, Deep Bass, Best Sp...",3.9,292
3,4,B00VE0PCAI,"Wireless Earbuds, Syllable D700 Bluetooth Wireless Headsets Hands-Free Calling in-line Volume Co...",3.2,190
4,5,B09FSW9FNH,Wireless Earbuds Bluetooth Ear Buds 35H Cycle Playtime Wireless Stereo Earphones for iPhone/Andr...,4.0,64


QUERY: comfortable earbuds for running with good battery life


,Rank,Product ID,Title,Rating,Reviews
0,1,B08H15SQ3H,Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stere...,4.0,76
1,2,B07NP3KSN2,"TREBLAB X5 - High-End Bluetooth Earbuds w/Beryllium Speakers - True HD Sound, Deep Bass, Best Sp...",3.9,292
2,3,B00VE0PCAI,"Wireless Earbuds, Syllable D700 Bluetooth Wireless Headsets Hands-Free Calling in-line Volume Co...",3.2,190
3,4,B00KO71XVO,Earhoox for EarPods - Compatible with iPhone 6/6+/5/5S/5C - Blue,4.1,183
4,5,B01N9K4XDW,Phaiser BHS-950 Bluetooth Headphones Headset Sport Earphones with Mic - Wireless Earbuds for Run...,4.0,231


QUERY: waterproof wireless earbuds for sports


,Rank,Product ID,Title,Rating,Reviews
0,1,B079L6WWD3,"Headphones, 2018 Wireless Earbuds for Sports Activities, Running, Gym. 8 Hour Battery, IPX7 Wate...",3.9,75
1,2,B0C778Z3RJ,"Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time-C...",3.9,80
2,3,B08V8YSRM3,"Wireless Bluetooth Headphone, Painless Wearing Headset with Mic for Cell Phone, Non Ear Plug Non...",3.1,144
3,4,B01L7UBOQ0,"Wireless Earbuds, Bluetooth Headphones V4.2 Mic Earpieces True Wireless Stereo Hands Free Call i...",3.8,60
4,5,B0154NJMOC,"Voxkin Universal Waterproof Case with Waterproof Earphone and Headphone Jack, Armband, Compass, ...",4.0,115


QUERY: iPhone case with good grip


,Rank,Product ID,Title,Rating,Reviews
0,1,B07482R6VS,Spigen Neo Hybrid Designed for Apple iPhone 8 Plus Case (2017) / Designed for iPhone 7 Plus Case...,4.5,72
1,2,B09VQ3RMXL,Smartish iPhone SE Slim Case - Gripmunk [Lightweight + Protective] Thin Cover for Apple iPhone S...,4.6,921
2,3,B00MV77G7A,Speck Products CandyShell Grip Case for iPhone 6/6S - Moss Green/Black,4.3,156
3,4,B01EN7WZG2,Speck Products CandyShell Grip Case for iPhone 6 Plus/6S Plus - Lipstick Pink/Jay Blue,4.6,1788
4,5,B01K094G9G,Speck Products CandyShell Grip Cell Phone Case for iPhone 7/6S/6 - Harbor Blue/Perwinkle Blue,4.4,58


## 4. Generated Recommendations

The retrieved products and their supporting information are provided to Gemini to generate recommendations for each evaluation query.

In [5]:
generated_answers = {}

for query in queries:
    results = retrieved_results[query]
    context = build_context(results)

    answer = generator.generate(
        query=query,
        context=context
    )

    generated_answers[query] = answer

    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)
    display(answer)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


QUERY: wireless earbuds with good battery life


'### Recommendations\n\n- **Product:** Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stereo Earphones for iPhone/Android (ID: B08H15SQ3H)\n  - **Why it matches:** The product claims extended playtime in its title, and multiple customer reviews highlight its long-lasting battery performance.\n  - **Key evidence:**\n    - *Metadata/Title:* Lists 35 hours of cycle playtime with the included charging case.\n    - *Customer Reviews:* Customers report that the battery "seems to last a long time," provides "extremely long" battery life, and allows listening for "hours at a time without needing to charge."\n  - **Considerations:** One customer reported receiving an initial defective unit that did not hold a charge for more than 10 minutes, though the replacement unit performed well.\n\n- **Product:** Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time (ID: B0C778Z3RJ)\n  - **Why it matches:** Offers high ad

QUERY: comfortable earbuds for running with good battery life


'### Recommendations\n\n- **Product:** Phaiser BHS-950 Bluetooth Headphones (ID: B01N9K4XDW)\n  - **Why it matches:** Designed specifically for active use and running, offering comfortable earhooks, sweat protection, and long battery life.\n  - **Key evidence:** \n    - *Metadata:* Features bullet-shaped Comply memory foam tips, Liquipel Nano Coating sweatproofing, and secure earhooks designed for rigorous workouts.\n    - *Customer Reviews:* Review 1 highlights that they are "comfortable and stay in place well even while running" with a battery life lasting "about seven hours." Review 3 notes the ear clips stay secure during long, sweaty runs.\n  - **Considerations:** Review 1 notes that bass performance is weak without Comply foam tips and mentions that Amazon grouped reviews for an older Phaiser model under this listing.\n\n- **Product:** TREBLAB X5 True Wireless Earbuds (ID: B07NP3KSN2)\n  - **Why it matches:** Features sports/running design with IPX4 waterproofing, along with posi

QUERY: waterproof wireless earbuds for sports


'### Recommendations\n\n- **Product:** Headphones, 2018 Wireless Earbuds for Sports Activities, Running, Gym (Product ID: B079L6WWD3)\n- **Why it matches:** This product explicitly satisfies all user requirements: it is a set of wireless earbuds featuring an IPX7 waterproof and sweatproof rating, designed specifically with an around-ear hold for sports, running, and gym activities.\n- **Key evidence:**\n  - *Metadata Specifications:* Features an IPX7 waterproof/sweatproof rating, wireless connectivity (with up to 10 meters range), ergonomic around-ear design for sports, and 6–8 hours of battery life.\n  - *Customer Feedback:* Multiple 5-star reviews note that the headphones "don\'t slide around" during workouts, offer good fitment and sound quality, and connect easily.\n- **Considerations:** The exact price is not provided in the metadata and cannot be verified.\n\n---\n\n### Not Recommended\n\n- **Product:** Wireless Earbuds, Bluetooth 5.2 Headphones (Product ID: B0C778Z3RJ)\n  - **Re

QUERY: iPhone case with good grip


'### Recommendations\n\n- **Product:** Smartish iPhone SE Slim Case - Gripmunk [Lightweight + Protective] Thin Cover for Apple iPhone SE 2022/2020 & iPhone 7/8 - Black Tie Affair (Product ID: B09VQ3RMXL)\n  - **Why it matches:** The case is specifically designed for high tactile grip, featuring high-grip textured sides and body to prevent drops.\n  - **Key evidence:** Product specifications list a "SUPER GRIPPY TEXTURE" and "HIGH-GRIP TEXTURED SIDES." Customer reviews overwhelmingly confirm the grip performance, describing a "great grippy feel" and noting that the textured grip holds well in the hand while still sliding into pockets without issue.\n  - **Considerations:** One reviewer noted that the TPU edges might stretch slightly over time, and another reported screen damage after dropping the phone from bed height onto a hard surface.\n\n- **Product:** Speck Products CandyShell Grip Cell Phone Case for iPhone 7/6S/6 - Harbor Blue/Perwinkle Blue (Product ID: B01K094G9G)\n  - **Why it

## 5. Grounding and Constraint Assessment

The generated recommendations are manually assessed for whether they are grounded in the retrieved product information and whether the user's stated requirements are addressed.

### Assessment Criteria

- **Grounded:** The recommendation and supporting claims are consistent with the retrieved product information.
- **Constraints handled:** The response appropriately addresses the requirements stated in the query.
- **Issues:** Important limitations, unsupported requirements, or conflicting evidence identified in the response.

In [6]:
for query, answer in generated_answers.items():
    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)
    display(answer)

QUERY: wireless earbuds with good battery life


'### Recommendations\n\n- **Product:** Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stereo Earphones for iPhone/Android (ID: B08H15SQ3H)\n  - **Why it matches:** The product claims extended playtime in its title, and multiple customer reviews highlight its long-lasting battery performance.\n  - **Key evidence:**\n    - *Metadata/Title:* Lists 35 hours of cycle playtime with the included charging case.\n    - *Customer Reviews:* Customers report that the battery "seems to last a long time," provides "extremely long" battery life, and allows listening for "hours at a time without needing to charge."\n  - **Considerations:** One customer reported receiving an initial defective unit that did not hold a charge for more than 10 minutes, though the replacement unit performed well.\n\n- **Product:** Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time (ID: B0C778Z3RJ)\n  - **Why it matches:** Offers high ad

QUERY: comfortable earbuds for running with good battery life


'### Recommendations\n\n- **Product:** Phaiser BHS-950 Bluetooth Headphones (ID: B01N9K4XDW)\n  - **Why it matches:** Designed specifically for active use and running, offering comfortable earhooks, sweat protection, and long battery life.\n  - **Key evidence:** \n    - *Metadata:* Features bullet-shaped Comply memory foam tips, Liquipel Nano Coating sweatproofing, and secure earhooks designed for rigorous workouts.\n    - *Customer Reviews:* Review 1 highlights that they are "comfortable and stay in place well even while running" with a battery life lasting "about seven hours." Review 3 notes the ear clips stay secure during long, sweaty runs.\n  - **Considerations:** Review 1 notes that bass performance is weak without Comply foam tips and mentions that Amazon grouped reviews for an older Phaiser model under this listing.\n\n- **Product:** TREBLAB X5 True Wireless Earbuds (ID: B07NP3KSN2)\n  - **Why it matches:** Features sports/running design with IPX4 waterproofing, along with posi

QUERY: waterproof wireless earbuds for sports


'### Recommendations\n\n- **Product:** Headphones, 2018 Wireless Earbuds for Sports Activities, Running, Gym (Product ID: B079L6WWD3)\n- **Why it matches:** This product explicitly satisfies all user requirements: it is a set of wireless earbuds featuring an IPX7 waterproof and sweatproof rating, designed specifically with an around-ear hold for sports, running, and gym activities.\n- **Key evidence:**\n  - *Metadata Specifications:* Features an IPX7 waterproof/sweatproof rating, wireless connectivity (with up to 10 meters range), ergonomic around-ear design for sports, and 6–8 hours of battery life.\n  - *Customer Feedback:* Multiple 5-star reviews note that the headphones "don\'t slide around" during workouts, offer good fitment and sound quality, and connect easily.\n- **Considerations:** The exact price is not provided in the metadata and cannot be verified.\n\n---\n\n### Not Recommended\n\n- **Product:** Wireless Earbuds, Bluetooth 5.2 Headphones (Product ID: B0C778Z3RJ)\n  - **Re

QUERY: iPhone case with good grip


'### Recommendations\n\n- **Product:** Smartish iPhone SE Slim Case - Gripmunk [Lightweight + Protective] Thin Cover for Apple iPhone SE 2022/2020 & iPhone 7/8 - Black Tie Affair (Product ID: B09VQ3RMXL)\n  - **Why it matches:** The case is specifically designed for high tactile grip, featuring high-grip textured sides and body to prevent drops.\n  - **Key evidence:** Product specifications list a "SUPER GRIPPY TEXTURE" and "HIGH-GRIP TEXTURED SIDES." Customer reviews overwhelmingly confirm the grip performance, describing a "great grippy feel" and noting that the textured grip holds well in the hand while still sliding into pockets without issue.\n  - **Considerations:** One reviewer noted that the TPU edges might stretch slightly over time, and another reported screen damage after dropping the phone from bed height onto a hard surface.\n\n- **Product:** Speck Products CandyShell Grip Cell Phone Case for iPhone 7/6S/6 - Harbor Blue/Perwinkle Blue (Product ID: B01K094G9G)\n  - **Why it

In [8]:
rag_assessment = pd.DataFrame([
    {
        "query": "iPhone",
        "grounded": "Yes",
        "constraints_handled": "Partially",
        "issues": "Retrieved products were dominated by accessories rather than iPhone smartphones, so the model correctly avoided making an unsupported recommendation."
    },
    {
        "query": "wireless earbuds with good battery life",
        "grounded": "Yes",
        "constraints_handled": "Yes",
        "issues": "Conflicting battery-life claims were identified for one product."
    },
    {
        "query": "comfortable earbuds for running with good battery life",
        "grounded": "Yes",
        "constraints_handled": "Yes",
        "issues": "Mixed evidence regarding running stability was identified for one product."
    },
    {
        "query": "waterproof wireless earbuds for sports",
        "grounded": "Yes",
        "constraints_handled": "Yes",
        "issues": "Only one of the five retrieved products fully matched; the model correctly filtered the others using evidence from metadata and reviews."
    },
    {
        "query": "iPhone case with good grip",
        "grounded": "Yes",
        "constraints_handled": "Yes",
        "issues": "Conflicting grip evidence was identified for one product."
    },
    {
        "query": "earbuds with good battery life but poor Bluetooth connectivity is unacceptable",
        "grounded": "Yes",
        "constraints_handled": "Partially",
        "issues": "The retrieved products provided evidence about battery life and Bluetooth performance, but did not consistently satisfy both requirements."
    }
])

rag_assessment

,query,grounded,constraints_handled,issues
0,iPhone,Yes,Partially,"Retrieved products were dominated by accessories rather than iPhone smartphones, so the model co..."
1,wireless earbuds with good battery life,Yes,Yes,Conflicting battery-life claims were identified for one product.
2,comfortable earbuds for running with good battery life,Yes,Yes,Mixed evidence regarding running stability was identified for one product.
3,waterproof wireless earbuds for sports,Yes,Yes,Only one of the five retrieved products fully matched; the model correctly filtered the others u...
4,iPhone case with good grip,Yes,Yes,Conflicting grip evidence was identified for one product.
5,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Yes,Partially,"The retrieved products provided evidence about battery life and Bluetooth performance, but did n..."


## 6. Failure Cases and Limitations

The RAG pipeline depends on the quality of the retrieved products and the information available in the product documents. Retrieval errors can lead to unsuitable products being passed to the generation stage. Complex constraints may also be difficult to verify when the retrieved metadata or customer reviews do not provide sufficient evidence.

In some cases, the model may correctly avoid making a recommendation when the retrieved evidence is insufficient. However, this does not resolve the underlying retrieval limitation.

The manual assessment in this notebook uses a small set of representative queries and is intended to examine grounding and constraint handling rather than provide a statistically rigorous evaluation.

## 7. Summary

The RAG pipeline generated recommendations grounded in the retrieved product information and customer reviews across the evaluated queries. The model generally handled user requirements by considering product metadata, customer feedback, conflicting evidence, and unsupported constraints. The evaluation also showed that generation quality remains dependent on retrieval quality, particularly for queries where the retrieved products do not sufficiently match the requested product type or multiple constraints.